In [41]:
import sys 
import os

# Add the parent directory to the path if it's not already there
if os.path.abspath('..') not in sys.path:
    sys.path.append(os.path.abspath('..'))

In [42]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [43]:
from ddql_optimal_execution import DDQL, MarketEnvironnement, MarketEnvironnementOld, Trainer, TWAP
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

In [44]:
QV = True  #next true, next false
Volume = False #next False, next false

In [45]:
env = MarketEnvironnement(initial_inventory=500, multi_episodes=True, QV=QV, Volume=Volume, data_path='../data/train')

In [46]:
agent = DDQL(state_size=env.state_size, initial_budget=env.initial_inventory, horizon=env.horizon)

Using cpu device


In [47]:
trainer = Trainer(agent, env, capacity=10000)

In [48]:
trainer.fill_exp_replay(max_steps=10000)

Filling experience replay buffer: 10002it [00:11, 838.20it/s]                                                           


In [49]:
trainer.pretrain(max_steps=100, batch_size=128)

Pretraining agent: 100%|██████████████████████████████████████████████████████████████| 100/100 [00:03<00:00, 27.65it/s]


In [50]:
trainer.agent.greediness = 0.9

In [51]:
trainer.train(max_steps=1000, batch_size=128)

Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:35<00:00, 28.11it/s]


In [58]:
def run_evaluation(trainer, test_env, twap_cls, n_episodes=100):
    pnl_ddql, pnl_twap = [], []
    n_ep = min(len(test_env.historical_data_series), n_episodes)
    random_ep = np.random.choice(np.arange(n_ep), size=n_ep, replace=True)

    for ep in random_ep:
        # TWAP
        test_env.swap_episode(ep)
        twap = twap_cls(initial_budget=test_env.initial_inventory, horizon=test_env.horizon)
        while not test_env.done:
            action = twap(test_env.state.copy())
            _ = test_env.step(action)
        pnl_twap.append(np.sum(test_env.pnl_for_episode))
        test_env.reset()

        # DDQL
        test_env.swap_episode(ep)
        while not test_env.done:
            action = trainer.agent(test_env.state.copy())
            _ = test_env.step(action)
        pnl_ddql.append(np.sum(test_env.pnl_for_episode))
        test_env.reset()

    return np.array(pnl_ddql), np.array(pnl_twap)


def print_comparison(label, pnl_ddql, pnl_twap):
    delta = pnl_ddql - pnl_twap
    glr   = delta[delta > 0].mean() / abs(delta[delta < 0].mean()) if (delta < 0).any() else float('inf')
    print(f"\n{'─'*45}")
    print(f"  {label}")
    print(f"{'─'*45}")
    print(f"  Mean   ΔPnL  : {delta.mean():.4f}")
    print(f"  Median ΔPnL  : {np.median(delta):.4f}")
    print(f"  Std    ΔPnL  : {delta.std():.4f}")
    print(f"  Win Prob      : {np.bincount(delta > 0)[1] / len(delta):.2%}")
    print(f"  GLR           : {glr:.4f}")
    print(f"{'─'*45}")


# ── build & train OLD env (original reward, lambda_risk=0) ───────────────────
print("Training OLD environment (original reward)...")
env_old = MarketEnvironnementOld(
    initial_inventory=500, multi_episodes=True,
    QV=True, Volume=False, data_path='../data/train'
)
agent_old  = DDQL(state_size=env_old.state_size, initial_budget=env_old.initial_inventory, horizon=env_old.horizon)
trainer_old = Trainer(agent_old, env_old, capacity=10000)
trainer_old.fill_exp_replay(max_steps=10000)
trainer_old.pretrain(max_steps=100, batch_size=128)
trainer_old.agent.greediness = 0.9
trainer_old.train(max_steps=1000, batch_size=128)

test_env_old = MarketEnvironnementOld(
    initial_inventory=500, multi_episodes=True,
    QV=True, Volume=False, data_path='../data/test'
)
pnl_ddql_old, pnl_twap_old = run_evaluation(trainer_old, test_env_old, TWAP)
print_comparison("OLD reward (lambda_risk = 0)", pnl_ddql_old, pnl_twap_old)


# ── build & train NEW env (Method B reward, lambda_risk=0.005) ───────────────
print("\nTraining NEW environment (Method B reward)...")
env_new = MarketEnvironnement(
    initial_inventory=500, multi_episodes=True,
    QV=True, Volume=False, data_path='../data/train',
    lambda_risk=10                              
)
agent_new   = DDQL(state_size=env_new.state_size, initial_budget=env_new.initial_inventory, horizon=env_new.horizon)
trainer_new = Trainer(agent_new, env_new, capacity=10000)
trainer_new.fill_exp_replay(max_steps=10000)
trainer_new.pretrain(max_steps=100, batch_size=128)
trainer_new.agent.greediness = 0.9
trainer_new.train(max_steps=1000, batch_size=128)

test_env_new = MarketEnvironnement(
    initial_inventory=500, multi_episodes=True,
    QV=True, Volume=False, data_path='../data/test',
    lambda_risk=10
)
pnl_ddql_new, pnl_twap_new = run_evaluation(trainer_new, test_env_new, TWAP)
print_comparison("NEW reward (Method B, lambda_risk=10)", pnl_ddql_new, pnl_twap_new)

Training OLD environment (original reward)...
Using cpu device


Filling experience replay buffer: 10003it [00:11, 850.52it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:35<00:00, 27.92it/s]



─────────────────────────────────────────────
  OLD reward (lambda_risk = 0)
─────────────────────────────────────────────
  Mean   ΔPnL  : 24977.9455
  Median ΔPnL  : 24130.5643
  Std    ΔPnL  : 34610.1718
  Win Prob      : 78.00%
  GLR           : 2.3256
─────────────────────────────────────────────

Training NEW environment (Method B reward)...
Using cpu device


Filling experience replay buffer: 10002it [00:11, 836.05it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:34<00:00, 28.95it/s]



─────────────────────────────────────────────
  NEW reward (Method B, lambda_risk=10)
─────────────────────────────────────────────
  Mean   ΔPnL  : 7634.5258
  Median ΔPnL  : 14030.4548
  Std    ΔPnL  : 56409.3732
  Win Prob      : 56.00%
  GLR           : 1.0940
─────────────────────────────────────────────


In [ ]:
import pandas as pd
import numpy as np
import os
os.makedirs('../figs', exist_ok=True)

tickers = ['JNJ', 'UNH', 'LLY', 'MDT', 'NVO', 'PFE', 'ABT', 'ABBV', 'DGX']
comparison_lambdas = [0, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.10]
all_results = []

for lam in comparison_lambdas:
    print(f"\n{'='*50}\nLambda = {lam}\n{'='*50}")
    
    # Choose environment based on lambda
    EnvClass = MarketEnvironnementOld if lam == 0 else MarketEnvironnement
    model_type = "Original (lambda=0)" if lam == 0 else f"Risk-Averse (lambda={lam})"

    for ticker in tickers:
        print(f"  Training {ticker}...")

        env_kwargs = dict(
            initial_inventory=500,
            multi_episodes=True,
            QV=True,
            Volume=False,
            data_path='../data/train'
        )
        if lam > 0:
            env_kwargs['lambda_risk'] = lam

        env = EnvClass(**env_kwargs)
        env.historical_data_series = [f for f in env.historical_data_series if ticker in f]
        env.swap_episode(0)

        agent = DDQL(state_size=env.state_size, initial_budget=env.initial_inventory, horizon=env.horizon)
        trainer = Trainer(agent, env, capacity=10000)
        trainer.fill_exp_replay(max_steps=10000)
        trainer.pretrain(max_steps=100, batch_size=128)
        trainer.agent.greediness = 0.9
        trainer.train(max_steps=1000, batch_size=128)

        # Evaluate on test
        test_kwargs = dict(
            initial_inventory=500,
            multi_episodes=True,
            QV=True,
            Volume=False,
            data_path='../data/test'
        )
        if lam > 0:
            test_kwargs['lambda_risk'] = lam

        test_env = EnvClass(**test_kwargs)
        test_env.historical_data_series = [f for f in test_env.historical_data_series if ticker in f]
        test_env.swap_episode(0)

        twap = TWAP(initial_budget=test_env.initial_inventory, horizon=test_env.horizon)
        pnl_twap, pnl_ddql = [], []

        n_episodes = min(len(test_env.historical_data_series), 100)
        random_ep = np.random.choice(np.arange(n_episodes), size=n_episodes, replace=True)

        for ep in random_ep:
            test_env.swap_episode(ep)
            while not test_env.done:
                test_env.step(twap(test_env.state.copy()))
            pnl_twap.append(np.sum(test_env.pnl_for_episode))
            test_env.reset()

            test_env.swap_episode(ep)
            while not test_env.done:
                test_env.step(trainer.agent(test_env.state.copy()))
            pnl_ddql.append(np.sum(test_env.pnl_for_episode))
            test_env.reset()

        pnl_ddql = np.array(pnl_ddql)
        pnl_twap = np.array(pnl_twap)
        delta = (pnl_ddql - pnl_twap) / np.abs(pnl_twap) * 10000
        delta = delta[np.abs(delta) < 500]

        glr = delta[delta > 0].mean() / abs(delta[delta < 0].mean()) if (delta < 0).any() else float('inf')

        all_results.append({
            "Model Type": model_type,
            "Lambda": lam,
            "Ticker": ticker,
            "Mean (bps)": round(delta.mean(), 2),
            "Median (bps)": round(np.median(delta), 2),
            "Std (bps)": round(delta.std(), 2),
            "P(win)": round((delta > 0).mean(), 4),
            "GLR": round(glr, 4)
        })
        print(f"    {ticker}: mean={delta.mean():.2f}bps  median={np.median(delta):.2f}bps  P(win)={(delta>0).mean()*100:.1f}%  GLR={glr:.2f}")

# Summary tables
results_df = pd.DataFrame(all_results)
results_df.to_csv('../figs/all_results.csv', index=False)

print("\n\n════ RESULTS BY LAMBDA (averaged across tickers) ════")
print(results_df.groupby('Lambda')[['Mean (bps)', 'Median (bps)', 'P(win)', 'GLR']].mean().to_string(float_format="{:.4f}".format))

print("\n\n════ RESULTS BY TICKER (averaged across lambdas) ════")
print(results_df.groupby('Ticker')[['Mean (bps)', 'Median (bps)', 'P(win)', 'GLR']].mean().to_string(float_format="{:.4f}".format))

Training Baseline OLD environment (original reward, lambda_risk=0)...
Using cpu device


Filling experience replay buffer: 10003it [00:11, 854.79it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:35<00:00, 28.09it/s]



Training NEW environment with lambda_risk = 0.1...
Using cpu device


Filling experience replay buffer: 10002it [00:12, 817.99it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:35<00:00, 28.36it/s]



Training NEW environment with lambda_risk = 0.25...
Using cpu device


...


Training NEW environment with lambda_risk = 1.2...
Using cpu device


Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:34<00:00, 28.95it/s]



Training NEW environment with lambda_risk = 1.5...
Using cpu device


Filling experience replay buffer: 10002it [00:12, 809.91it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:34<00:00, 29.05it/s]




════ MULTI-LAMBDA ARCHITECTURE COMPARISON ════
                                  Mean ΔPnL  Median ΔPnL   Std ΔPnL  Win Prob    GLR
Model Type           Lambda Risk                                                    
Original Paper (Old) 0.00         8881.2455   -4335.4978 65624.6757    0.4700 1.6410
Risk-Averse (New)    0.10          195.1257   -4462.4897 75972.4452    0.4700 1.1349
                     0.25        -2244.4617   -2243.6478 70369.5380    0.4900 0.9642
                     0.50        12957.1970   11711.7304 56105.8734    0.6200 1.1422
                     0.80         2241.7766    1641.9436 26427.7062    0.5200 1.1552
                     0.90         9747.4138   14746.2439 59809.7203    0.5900 1.0407
                     1.00        -6182.8593    -448.7647 55431.1328    0.5000 0.7567
                     1.10        12483.8391    9141.6577 45230.9644    0.5800 1.4730
                     1.20        -5285.2732  -14237.8321 88429.5338    0.4400 1.1067
                

In [62]:
import pandas as pd
import numpy as np

# Define the risk values you want to test the new environment against
comparison_lambdas = [0.0001, 0.001, 0.01, 0.1]
comparison_results = []

# ── 1. Train & Evaluate the Baseline OLD Env ONCE (Since lambda is always 0) ──
print("Training Baseline OLD environment (original reward, lambda_risk=0)...")
env_old = MarketEnvironnementOld(
    initial_inventory=500, multi_episodes=True,
    QV=True, Volume=False, data_path='../data/train'
)
agent_old = DDQL(state_size=env_old.state_size, initial_budget=env_old.initial_inventory, horizon=env_old.horizon)
trainer_old = Trainer(agent_old, env_old, capacity=10000)
trainer_old.fill_exp_replay(max_steps=10000)
trainer_old.pretrain(max_steps=100, batch_size=128)
trainer_old.agent.greediness = 0.9
trainer_old.train(max_steps=1000, batch_size=128)

test_env_old = MarketEnvironnementOld(
    initial_inventory=500, multi_episodes=True,
    QV=True, Volume=False, data_path='../data/test'
)
pnl_ddql_old, pnl_twap_old = run_evaluation(trainer_old, test_env_old, TWAP)

delta_old = pnl_ddql_old - pnl_twap_old
glr_old = delta_old[delta_old > 0].mean() / abs(delta_old[delta_old < 0].mean()) if (delta_old < 0).any() else float('inf')

# Save the baseline stats to insert into our final table
comparison_results.append({
    "Model Type": "Original Paper (Old)",
    "Lambda Risk": 0.0,
    "Mean ΔPnL": delta_old.mean(),
    "Median ΔPnL": np.median(delta_old),
    "Std ΔPnL": delta_old.std(),
    "Win Prob": np.bincount(delta_old > 0)[1] / len(delta_old),
    "GLR": glr_old
})


# ── 2. Loop Through Different Lambda Values for the NEW Env ───────────────────
for lam in comparison_lambdas:
    print(f"\nTraining NEW environment with lambda_risk = {lam}...")
    env_new = MarketEnvironnement(
        initial_inventory=500, multi_episodes=True,
        QV=True, Volume=False, data_path='../data/train',
        lambda_risk=lam                        
    )
    agent_new = DDQL(state_size=env_new.state_size, initial_budget=env_new.initial_inventory, horizon=env_new.horizon)
    trainer_new = Trainer(agent_new, env_new, capacity=10000)
    trainer_new.fill_exp_replay(max_steps=10000)
    trainer_new.pretrain(max_steps=100, batch_size=128)
    trainer_new.agent.greediness = 0.9
    trainer_new.train(max_steps=1000, batch_size=128)

    test_env_new = MarketEnvironnement(
        initial_inventory=500, multi_episodes=True,
        QV=True, Volume=False, data_path='../data/test',
        lambda_risk=lam
    )
    pnl_ddql_new, pnl_twap_new = run_evaluation(trainer_new, test_env_new, TWAP)
    
    delta_new = pnl_ddql_new - pnl_twap_new
    glr_new = delta_new[delta_new > 0].mean() / abs(delta_new[delta_new < 0].mean()) if (delta_new < 0).any() else float('inf')
    
    comparison_results.append({
        "Model Type": "Risk-Averse (New)",
        "Lambda Risk": lam,
        "Mean ΔPnL": delta_new.mean(),
        "Median ΔPnL": np.median(delta_new),
        "Std ΔPnL": delta_new.std(),
        "Win Prob": np.bincount(delta_new > 0)[1] / len(delta_new),
        "GLR": glr_new
    })

# ── 3. Display the Master Head-to-Head Summary Table ──────────────────────────
summary_df = pd.DataFrame(comparison_results).set_index(["Model Type", "Lambda Risk"])
print("\n\n════ MULTI-LAMBDA ARCHITECTURE COMPARISON ════")
print(summary_df.to_string(float_format="{:.4f}".format))

Training Baseline OLD environment (original reward, lambda_risk=0)...
Using cpu device


Filling experience replay buffer: 10002it [00:12, 808.35it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:35<00:00, 27.81it/s]



Training NEW environment with lambda_risk = 0.0001...
Using cpu device


Filling experience replay buffer: 10002it [00:12, 820.15it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:34<00:00, 28.89it/s]



Training NEW environment with lambda_risk = 0.001...
Using cpu device


...


Training NEW environment with lambda_risk = 0.01...
Using cpu device


Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:34<00:00, 29.23it/s]



Training NEW environment with lambda_risk = 0.1...
Using cpu device


Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:33<00:00, 29.45it/s]




════ MULTI-LAMBDA ARCHITECTURE COMPARISON ════
                                  Mean ΔPnL  Median ΔPnL   Std ΔPnL  Win Prob    GLR
Model Type           Lambda Risk                                                    
Original Paper (Old) 0.0000      -9671.8045  -18544.8829 73410.1652    0.4300 0.9579
Risk-Averse (New)    0.0001      23610.9039   17598.0039 85664.3361    0.6200 1.2877
                     0.0010      29699.2399   25853.1958 56147.6166    0.7000 1.9927
                     0.0100      15964.6435  -14662.5088 82399.7445    0.4500 1.9877
                     0.1000       7302.2456   15462.2660 45061.5589    0.6300 0.8840


In [66]:
import pandas as pd
import numpy as np

# Define the risk values you want to test the new environment against
comparison_lambdas = [0.00001, 0.00005, 0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.2, 0.5, 1]
comparison_results = []

# ── 1. Train & Evaluate the Baseline OLD Env ONCE (Since lambda is always 0) ──
print("Training Baseline OLD environment (original reward, lambda_risk=0)...")
env_old = MarketEnvironnementOld(
    initial_inventory=500, multi_episodes=True,
    QV=True, Volume=False, data_path='../data/train'
)
agent_old = DDQL(state_size=env_old.state_size, initial_budget=env_old.initial_inventory, horizon=env_old.horizon)
trainer_old = Trainer(agent_old, env_old, capacity=10000)
trainer_old.fill_exp_replay(max_steps=10000)
trainer_old.pretrain(max_steps=100, batch_size=128)
trainer_old.agent.greediness = 0.9
trainer_old.train(max_steps=1000, batch_size=128)

test_env_old = MarketEnvironnementOld(
    initial_inventory=500, multi_episodes=True,
    QV=True, Volume=False, data_path='../data/test'
)
pnl_ddql_old, pnl_twap_old = run_evaluation(trainer_old, test_env_old, TWAP)

delta_old = pnl_ddql_old - pnl_twap_old
glr_old = delta_old[delta_old > 0].mean() / abs(delta_old[delta_old < 0].mean()) if (delta_old < 0).any() else float('inf')

# Save the baseline stats to insert into our final table
comparison_results.append({
    "Model Type": "Original Paper (Old)",
    "Lambda Risk": 0.0,
    "Mean ΔPnL": delta_old.mean(),
    "Median ΔPnL": np.median(delta_old),
    "Std ΔPnL": delta_old.std(),
    "Win Prob": np.bincount(delta_old > 0)[1] / len(delta_old),
    "GLR": glr_old
})


# ── 2. Loop Through Different Lambda Values for the NEW Env ───────────────────
for lam in comparison_lambdas:
    print(f"\nTraining NEW environment with lambda_risk = {lam}...")
    env_new = MarketEnvironnement(
        initial_inventory=500, multi_episodes=True,
        QV=True, Volume=False, data_path='../data/train',
        lambda_risk=lam                        
    )
    agent_new = DDQL(state_size=env_new.state_size, initial_budget=env_new.initial_inventory, horizon=env_new.horizon)
    trainer_new = Trainer(agent_new, env_new, capacity=10000)
    trainer_new.fill_exp_replay(max_steps=10000)
    trainer_new.pretrain(max_steps=100, batch_size=128)
    trainer_new.agent.greediness = 0.9
    trainer_new.train(max_steps=1000, batch_size=128)

    test_env_new = MarketEnvironnement(
        initial_inventory=500, multi_episodes=True,
        QV=True, Volume=False, data_path='../data/test',
        lambda_risk=lam
    )
    pnl_ddql_new, pnl_twap_new = run_evaluation(trainer_new, test_env_new, TWAP)
    
    delta_new = pnl_ddql_new - pnl_twap_new
    glr_new = delta_new[delta_new > 0].mean() / abs(delta_new[delta_new < 0].mean()) if (delta_new < 0).any() else float('inf')
    
    comparison_results.append({
        "Model Type": "Risk-Averse (New)",
        "Lambda Risk": lam,
        "Mean ΔPnL": delta_new.mean(),
        "Median ΔPnL": np.median(delta_new),
        "Std ΔPnL": delta_new.std(),
        "Win Prob": np.bincount(delta_new > 0)[1] / len(delta_new),
        "GLR": glr_new
    })

# ── 3. Display the Master Head-to-Head Summary Table ──────────────────────────
summary_df = pd.DataFrame(comparison_results).set_index(["Model Type", "Lambda Risk"])
print("\n\n════ MULTI-LAMBDA ARCHITECTURE COMPARISON ════")
print(summary_df.to_string(float_format="{:.4f}".format))

Training Baseline OLD environment (original reward, lambda_risk=0)...
Using cpu device


Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:35<00:00, 28.02it/s]



Training NEW environment with lambda_risk = 1e-05...
Using cpu device


Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:34<00:00, 28.61it/s]



Training NEW environment with lambda_risk = 5e-05...
Using cpu device


...


Training NEW environment with lambda_risk = 0.5...
Using cpu device


Filling experience replay buffer: 10003it [00:12, 803.81it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:35<00:00, 28.35it/s]



Training NEW environment with lambda_risk = 1...
Using cpu device


Filling experience replay buffer: 10002it [00:12, 793.24it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:34<00:00, 28.89it/s]




════ MULTI-LAMBDA ARCHITECTURE COMPARISON ════
                                   Mean ΔPnL  Median ΔPnL   Std ΔPnL  Win Prob    GLR
Model Type           Lambda Risk                                                     
Original Paper (Old) 0.00000      23009.7239   18257.0735 76635.3771    0.6300 1.3339
Risk-Averse (New)    0.00001      15170.3886   15058.2561 49908.1470    0.6100 1.4839
                     0.00005     -10082.7837  -20186.0769 70963.9343    0.3300 1.3966
                     0.00010       7197.7933    8626.6351 50503.3682    0.5700 1.0832
                     0.00050       3683.0656  -13305.0663 78696.0788    0.4400 1.4343
                     0.00100     -13007.1529  -24585.8523 76776.7567    0.3800 1.0738
                     0.00500      -7405.6792  -21569.9397 79018.2299    0.3900 1.2352
                     0.01000      12714.0260   18923.8055 78444.3812    0.5600 1.1725
                     0.05000      10992.5524   11309.0077 79733.5037    0.5400 1.1924
     

In [53]:
import pandas as pd

lambda_grid = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
results = []

for lam in lambda_grid:
    print(f"\n── lambda_risk = {lam} ──────────────────────────")

    # Train
    env_lam = MarketEnvironnement(
        initial_inventory=500, multi_episodes=True,
        QV=True, Volume=False,
        data_path='../data/train',
        lambda_risk=lam
    )
    agent_lam   = DDQL(state_size=env_lam.state_size, initial_budget=env_lam.initial_inventory, horizon=env_lam.horizon)
    trainer_lam = Trainer(agent_lam, env_lam, capacity=10000)
    trainer_lam.fill_exp_replay(max_steps=10000)
    trainer_lam.pretrain(max_steps=100, batch_size=128)
    trainer_lam.agent.greediness = 0.9
    trainer_lam.train(max_steps=1000, batch_size=128)

    # Evaluate
    test_env_lam = MarketEnvironnement(
        initial_inventory=500, multi_episodes=True,
        QV=True, Volume=False,
        data_path='../data/test',
        lambda_risk=lam                    # pass same lambda at test time
    )
    pnl_ddql_lam, pnl_twap_lam = run_evaluation(trainer_lam, test_env_lam, TWAP)

    delta = pnl_ddql_lam - pnl_twap_lam
    glr   = delta[delta > 0].mean() / abs(delta[delta < 0].mean()) if (delta < 0).any() else float('inf')

    row = {
        "lambda_risk"  : lam,
        "mean_delta"   : delta.mean(),
        "median_delta" : np.median(delta),
        "std_delta"    : delta.std(),
        "win_prob"     : np.bincount(delta > 0)[1] / len(delta),
        "glr"          : glr,
        "pnl_variance": np.var(pnl_ddql_lam),   # lower = less risky
        "sharpe_proxy": delta.mean() / delta.std(),  # mean outperformance per unit of variability
    }
    results.append(row)
    print(f"  Mean ΔPnL={row['mean_delta']:.4f}  Win={row['win_prob']:.2%}  GLR={row['glr']:.4f}  Std={row['std_delta']:.2f}  Sharpe={row['sharpe_proxy']:.4f}  PnL Var={row['pnl_variance']:.2f}")

# ── summary table ─────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results).set_index("lambda_risk")
print("\n\n════ LAMBDA GRID SEARCH SUMMARY ════")
print(results_df.to_string(float_format="{:.4f}".format))

# pick best by win probability
best_lam = results_df["win_prob"].idxmax()
print(f"\n→ Best lambda_risk by win prob: {best_lam}")


── lambda_risk = 0.05 ──────────────────────────
Using cpu device


Filling experience replay buffer: 10001it [00:13, 758.59it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:35<00:00, 27.91it/s]


  Mean ΔPnL=12974.5983  Win=49.00%  GLR=1.5767  Std=80049.26  Sharpe=0.1621  PnL Var=31876399139.59

── lambda_risk = 0.1 ──────────────────────────
Using cpu device


Filling experience replay buffer: 10003it [00:12, 812.36it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:35<00:00, 28.40it/s]


  Mean ΔPnL=7771.8035  Win=54.00%  GLR=1.3053  Std=52124.88  Sharpe=0.1491  PnL Var=27972385811.48

── lambda_risk = 0.15 ──────────────────────────
Using cpu device


...

  Mean ΔPnL=7618.9555  Win=54.00%  GLR=1.2338  Std=54276.47  Sharpe=0.1404  PnL Var=24661417323.65

── lambda_risk = 0.25 ──────────────────────────
Using cpu device


Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:35<00:00, 28.36it/s]


  Mean ΔPnL=3771.5756  Win=60.00%  GLR=0.7677  Std=66809.79  Sharpe=0.0565  PnL Var=14101908698.79

── lambda_risk = 0.3 ──────────────────────────
Using cpu device


Filling experience replay buffer: 10003it [00:12, 787.70it/s]                                                           
Training agent: 100%|███████████████████████████████████████████████████████████████| 1000/1000 [00:35<00:00, 28.56it/s]


  Mean ΔPnL=788.6142  Win=46.00%  GLR=1.2184  Std=51867.69  Sharpe=0.0152  PnL Var=27183328242.13


════ LAMBDA GRID SEARCH SUMMARY ════
             mean_delta  median_delta  std_delta  win_prob    glr     pnl_variance  sharpe_proxy
lambda_risk                                                                                     
0.05         12974.5983     -365.9947 80049.2563    0.4900 1.5767 31876399139.5871        0.1621
0.10          7771.8035     1853.6095 52124.8764    0.5400 1.3053 27972385811.4843        0.1491
0.15          8049.8433     1827.3606 66341.5668    0.5200 1.2611 18290096596.9141        0.1213
0.20          7618.9555     8578.2184 54276.4708    0.5400 1.2338 24661417323.6524        0.1404
0.25          3771.5756    11293.8112 66809.7887    0.6000 0.7677 14101908698.7894        0.0565
0.30           788.6142    -4211.1778 51867.6948    0.4600 1.2184 27183328242.1309        0.0152

→ Best lambda_risk by win prob: 0.25


In [54]:
pnl_twap = []
pnl_ddql = []

pnl_ddql, pnl_twap = np.array(pnl_ddql), np.array(pnl_twap)

In [57]:
pnl_ddql_sum = pnl_ddql
pnl_twap_sum = pnl_twap

delta_pnl = (pnl_ddql_sum - pnl_twap_sum) / pnl_twap_sum



GLR = - delta_pnl[delta_pnl > 0].mean()/  delta_pnl[delta_pnl < 0].mean()

prob_win = np.bincount(delta_pnl > 0)[1] / len(delta_pnl)


/var/folders/rd/61g1g51n6t9cq2twgssm9y6c0000gn/T/ipykernel_23422/1987651981.py:8: RuntimeWarning: Mean of empty slice.
  GLR = - delta_pnl[delta_pnl > 0].mean()/  delta_pnl[delta_pnl < 0].mean()
/Users/ishaniagarwal/Library/Python/3.9/lib/python/site-packages/numpy/core/_methods.py:190: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


IndexError: index 1 is out of bounds for axis 0 with size 0

In [ ]:
for QV in [True, ]:
    for Volume in [False, ]:
        env = MarketEnvironnement(initial_inventory=500, multi_episodes=True, QV=QV, Volume=Volume, data_path='../data/train', quadratic_penalty_coefficient=0)
        agent = DDQL(state_size=env.state_size, initial_budget=env.initial_inventory, horizon=env.horizon, quadratic_penalty_coefficient=0)
        trainer = Trainer(agent, env, capacity=10000)
        trainer.fill_exp_replay(max_steps=10000)
        trainer.pretrain(max_steps=100, batch_size=128)
        trainer.agent.greediness = 0.9
        trainer.train(max_steps=1000, batch_size=128)



        #test the agent
        for data_path in ['train', 'test']:
            test_env = MarketEnvironnement(initial_inventory=500, multi_episodes=True, QV=QV, Volume=Volume, data_path=f'../data/{data_path}', quadratic_penalty_coefficient=0)

            twap = TWAP(initial_budget=test_env.initial_inventory, horizon=test_env.horizon)

            pnl_twap = []
            pnl_ddql = []


            n_episodes = min(len(test_env.historical_data_series), 100)

            random_ep = np.random.choice(np.arange(n_episodes), size=n_episodes, replace=True)

            for ep in random_ep:
                test_env.swap_episode(ep)
                _pnl_twap = [0]
                while not test_env.done:
                    current_state = test_env.state.copy()
                    action = twap(current_state)
                    _ = test_env.step(action)
                    
                pnl_twap.append(test_env.pnl_for_episode + [test_env.state['Price']*test_env.state['inventory'] - test_env.quadratic_penalty_coefficient*(test_env.state['inventory']/test_env.initial_inventory)**2 / test_env.horizon])

                test_env.reset()
                
                _pnl_ddql = [0]
                while not test_env.done:
                    current_state = test_env.state.copy()
                    action = trainer.agent(current_state)
                    _ = test_env.step(action)
                pnl_ddql.append(test_env.pnl_for_episode + [test_env.state['Price']*test_env.state['inventory'] - test_env.quadratic_penalty_coefficient*(test_env.state['inventory']/test_env.initial_inventory)**2 / test_env.horizon])

            pnl_ddql, pnl_twap = np.array(pnl_ddql), np.array(pnl_twap)

            pnl_ddql_sum = pnl_ddql.sum(axis=1)
            pnl_twap_sum = pnl_twap.sum(axis=1)

            delta_pnl = (pnl_ddql_sum - pnl_twap_sum)/ pnl_twap_sum

            GLR = - delta_pnl[delta_pnl > 0].mean()/  delta_pnl[delta_pnl < 0].mean()

            prob_win = np.bincount(delta_pnl > 0)[1] / len(delta_pnl)
            sns.histplot(delta_pnl, kde=True, stat='frequency')
            #change font$

            title = "\n".join([r'$\Delta$P&L', 'TIPTr' + 'V'*Volume+'QV'*QV])
            plt.title(title, fontdict={'fontfamily': 'times new roman'}, x=0.5, y=1, fontsize=20)
            x, y = plt.xlim()[0]*.9, plt.ylim()[1] * .7
            plt.text(x=x, y=y, s="\n".join([r'$\mathbb{P}(\Delta{P&L}'+r'>0 ) = {:.2f}$'.format(prob_win) , 
                        r'$Mean = {:.2f}$'.format(delta_pnl.mean()), r'$Std = {:.2f}$'.format(delta_pnl.std()), r'$Median = {:.2f}$'.format(np.median(delta_pnl)), r' $GLR = {:.2f}$'.format(GLR)]), fontsize=12, fontdict={'fontfamily': 'times new roman'})

            #save with ransparent background

            plt.savefig(title.replace("\n", "_").replace("$", "").replace("\\", "").replace("&", "n") +f'_{data_path}.png', dpi=300, bbox_inches='tight',transparent=True)
            plt.show()

In [ ]:
sns.histplot(pnl_twap.sum(axis=1), label='ddql', kde=True, stat='density', color='tab:red', alpha=.5)
sns.histplot(pnl_ddql.sum(axis=1), label='twap', kde=True, stat='density', color='tab:blue', alpha=.5)
plt.legend()
plt.title('Distribution of PnL')
plt.xlabel('PnL')

In [ ]:
np.mean(pnl_twap.sum(axis=1))-np.mean(pnl_ddql.sum(axis=1))

In [ ]:
grid = {
    "QV": [True, False],
    "Volume": [False],

}

data = dict()

for QV in grid["QV"]:
    for Volume in grid["Volume"]:
        env = MarketEnvironnement(initial_inventory=100, multi_episodes=True, QV=QV, Volume=Volume, data_path='../data/train')
        agent = DDQL(state_size=env.state_size, initial_budget=env.initial_inventory, horizon=env.horizon)
        trainer = Trainer(agent, env, capacity=10000)
        trainer.fill_exp_replay(max_steps=10000)
        trainer.pretrain(max_steps=1000, batch_size=128)
        trainer.train(max_steps=10000, batch_size=128)
        env.reset()
        twap = TWAP(initial_budget=env.initial_inventory, horizon=env.horizon)
        pnl_twap = []
        pnl_ddql = []
        random_ep = np.random.choice(np.arange(len(env.historical_data_series)), 100)
        for ep in random_ep:
            env.swap_episode(ep)
            _pnl_twap = [0]
            while not env.done:
                current_state = env.state.copy()
                action = twap(current_state)
                _ = env.step(action)
                _pnl_twap.append(env.state['Price']*action)
            pnl_twap.append(_pnl_twap)
            env.reset()
            _pnl_ddql = [0]
            while not env.done:
                current_state = env.state.copy()
                action = trainer.agent(current_state)
                _ = env.step(action)
                _pnl_ddql.append(env.state['Price']*action)
            pnl_ddql.append(_pnl_ddql)

        pnl_ddql, pnl_twap = np.array(pnl_ddql), np.array(pnl_twap)
        data[(QV, Volume)] = (pnl_ddql, pnl_twap)



In [ ]:
import pickle

with open('data.pkl', 'wb') as f:
    pickle.dump(data, f)

In [ ]:
sns.set_palette('pastel')
sns.set_style('whitegrid')

In [ ]:
for i, (QV, Volume) in enumerate(data):
    pnl_ddql, pnl_twap = data[(QV, Volume)]
    
    confidence_level_2 = .99
    plt.plot(pnl_ddql.mean(axis=0), label='ddql', color='tab:blue')
    plt.fill_between(range(len(pnl_ddql.mean(axis=0))), pnl_ddql.mean(axis=0)-confidence_level_2*pnl_ddql.std(axis=0)/np.sqrt(len(pnl_ddql)), pnl_ddql.mean(axis=0)+confidence_level_2*pnl_ddql.std(axis=0)/np.sqrt(len(pnl_ddql)), alpha=.1,   color='tab:blue')

    plt.plot(pnl_twap.mean(axis=0), label='twap', color='tab:red')
    plt.fill_between(range(len(pnl_twap.mean(axis=0))), pnl_twap.mean(axis=0)-confidence_level_2*pnl_twap.std(axis=0)/np.sqrt(len(pnl_twap)), pnl_twap.mean(axis=0)+confidence_level_2*pnl_twap.std(axis=0)/np.sqrt(len(pnl_twap)), alpha=.1, color='tab:red')
    plt.legend()
    plt.xticks(
        #rotation=45,
        ticks=range(5), labels=[f"{i}" for i in range(5)]
    )
    plt.title(f'Profits with normalized prices (with 99% confidence interval)\n QV={QV}, Volume={Volume}')
    
    plt.ylabel('PnL')
    plt.savefig(f'pnl_QV_{QV}_Volume_{Volume}.png')
    plt.show()

In [ ]:
for key in data:
    print(key, np.mean(data[key][0].sum(axis=1))-np.mean(data[key][1].sum(axis=1)))
    #plt.hist(data[key][0].sum(axis=1), label='ddql', alpha=.5, color='tab:blue')
    #plt.axvline(np.mean(data[key][0].sum(axis=1)), color='tab:blue')
    #plt.hist(data[key][1].sum(axis=1), label='twap', alpha=.5, color='tab:red')
    #plt.axvline(np.mean(data[key][1].sum(axis=1)), color='tab:red')

    pnl_twap, pnl_ddql = data[key]

    sns.histplot(pnl_twap.sum(axis=1), label='ddql', kde=True, stat='probability', color='tab:red', alpha=.5)
    sns.histplot(pnl_ddql.sum(axis=1), label='twap', kde=True, stat='probability', color='tab:blue', alpha=.5)
    #plt.legend()
    #plt.title('')
    plt.xlabel('PnL')

    plt.legend()
    plt.title(f'Distribution of PnL \nQV={key[0]}, Volume={key[1]}, Pnl Spread = {np.mean(data[key][0].sum(axis=1))-np.mean(data[key][1].sum(axis=1)):.2f}')
    plt.savefig(f'../figs/QV_{key[0]}_Volume_{key[1]}_hists.png')
    plt.show()


In [ ]:
#for i in range(100):
#    fake_data(start="2022-01-01 11:00:01", end="2022-01-01 12:00:00").to_csv(f"../data/fake_data_{i}.csv")
#    fake_data(start="2022-01-01 12:00:01", end="2022-01-01 13:00:00").to_csv(f"../data/fake_data_{i+100}.csv")
#    fake_data(start="2022-01-01 13:00:01", end="2022-01-01 14:00:00").to_csv(f"../data/fake_data_{i+200}.csv")AA

In [ ]:
# healthcare data with old reward function

import numpy as np
import matplotlib.pyplot as plt

# 1. Initialize the original paper's environment (Risk-Neutral baseline)
print("Initializing the original risk-neutral paper environment...")
env_old = MarketEnvironnementOld(
    initial_inventory=500, 
    multi_episodes=True, 
    QV=True, 
    Volume=False, 
    data_path='../data/train'
)

# 2. Build the Agent and Trainer structures
agent_old = DDQL(
    state_size=env_old.state_size, 
    initial_budget=env_old.initial_inventory, 
    horizon=env_old.horizon
)
trainer_old = Trainer(agent_old, env_old, capacity=10000)

# 3. Replay Buffer & Pretraining steps
print("Filling experience replay buffer...")
trainer_old.fill_exp_replay(max_steps=10000)

print("Pretraining the baseline agent...")
trainer_old.pretrain(max_steps=100, batch_size=128)

# 4. Train the risk-neutral agent
trainer_old.agent.greediness = 0.9
print("Training the baseline agent (1000 steps)...")
trainer_old.train(max_steps=1000, batch_size=128)
print("Baseline model training complete!\n")

# 5. Evaluate the baseline on the unseen Test data
print("Evaluating the risk-neutral paper baseline on test data...")
test_env_old = MarketEnvironnementOld(
    initial_inventory=500, 
    multi_episodes=True, 
    QV=True, 
    Volume=False, 
    data_path='../data/test'
)

pnl_twap_old = []
pnl_ddql_old = []
n_episodes = len(test_env_old.historical_data_series)

for ep in range(n_episodes):
    # Test TWAP Baseline
    test_env_old.swap_episode(ep)
    twap = TWAP(initial_budget=test_env_old.initial_inventory, horizon=test_env_old.horizon)
    
    while not test_env_old.done:
        current_state = test_env_old.state.copy()
        action = twap(current_state)
        _ = test_env_old.step(action)
        
    # Directly grab the total accumulated P&L for the entire episode
    pnl_twap_old.append(np.sum(test_env_old.pnl_for_episode))
    test_env_old.reset()
    
    # Test DDQL Risk-Neutral Agent
    test_env_old.swap_episode(ep)
    while not test_env_old.done:
        current_state = test_env_old.state.copy()
        action = trainer_old.agent(current_state) 
        _ = test_env_old.step(action)
        
    pnl_ddql_old.append(np.sum(test_env_old.pnl_for_episode))
    test_env_old.reset()

# 6. Compute and display summary stats cleanly
sum_ddql = np.array(pnl_ddql_old)
sum_twap = np.array(pnl_twap_old)
delta_pnl_old = sum_ddql - sum_twap

print(f"--- Original Paper Baseline Results (Healthcare Data) ---")
print(f"Mean Out-performance vs TWAP: {np.mean(delta_pnl_old):.4f}")
print(f"Median Out-performance vs TWAP: {np.median(delta_pnl_old):.4f}")
print(f"Win Probability P(ΔPnL > 0): {np.sum(delta_pnl_old > 0) / len(delta_pnl_old) * 100:.2f}%\n")